<a href="https://colab.research.google.com/github/Yingyan-Chen/26SUM-DRL/blob/main/Two_Envelopes_Paradox_%26_CLT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# # Assignment: The Two Envelopes Paradox & The Central Limit Theorem
#
# Welcome to this computational statistics and reinforcement learning prerequisite assignment!
#
# ## The Setup
# You are presented with two indistinguishable envelopes. You are told that one contains exactly **twice as much money** as the other.
# You randomly select one envelope. Before you open it, you are offered the chance to swap your envelope for the other one.
#
# ## The Paradox
# The naive reasoning goes like this:
# Let the amount in your current envelope be $A$.
# The other envelope contains either $A/2$ or $2A$, with a $50\%$ chance of each.
# The expected value of swapping is:
# $\mathbb{E}[\text{Swap}] = 0.5 \times (A/2) + 0.5 \times (2A) = 0.25A + 1A = 1.25A$
#
# Since $1.25A > A$, it seems you should *always* swap! But wait, by symmetry, the envelopes are identical. How can swapping always be advantageous?
#
# In this assignment, you will:
# 1. Build a simulation to test whether the expected value of swapping is actually higher.
# 2. Rewrite the simulation in **JAX** to run millions of parallel environments.
# 3. Use your massive JAX dataset to numerically demonstrate the **Central Limit Theorem (CLT)**.

# %%
# Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from scipy.stats import norm

# Set plot styling
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

# ## Part 1: Simulating the Paradox
#
# To break the paradox, we must realize that an unconditional uniform prior over all positive real numbers $(0, \infty)$ is a mathematical impossibility (an *improper prior*).
#
# Let's ground this in reality. We will draw the base amount $X$ from a Uniform distribution between $\$1$ and $\$100$.
# One envelope gets $X$, the other gets $2X$.
#
# ### Question 1: Implement the Game Engine
# Complete the function below to simulate `num_trials` games. For each game, randomly assign the envelopes, randomly pick one, and return the rewards for the `Keep` strategy versus the `Swap` strategy.


In [ ]:

def simulate_two_envelopes_numpy(num_trials=100000):
    """
    Simulates the two envelopes problem.
    """
    # 1. Generate the base amounts X ~ Uniform(1, 100)
    # TODO: Generate base amounts
    base_amounts = np.random.uniform(1, 100, size=num_trials)

    # 2. Create the two envelopes: Envelope A has X, Envelope B has 2X
    env_A = base_amounts
    env_B = 2 * base_amounts

    # 3. Randomly select an envelope (0 means pick A, 1 means pick B)
    # TODO: Generate random choices (0 or 1)
    choices = np.random.randint(0, 2, size=num_trials)

    # 4. Calculate the reward for keeping
    # If choice is 0, we keep env_A. If 1, we keep env_B.
    # TODO: Implement the keeping logic
    keep_rewards = np.where(choices == 0, env_A, env_B)

    # 5. Calculate the reward for swapping
    # If we swap, we get the opposite of what we chose!
    # TODO: Implement the swapping logic
    swap_rewards = np.where(choices == 0, env_B, env_A)

    return keep_rewards, swap_rewards

# Run the simulation
keep_res, swap_res = simulate_two_envelopes_numpy(100000)

print(f"Expected Return (Keep): ${np.mean(keep_res):.2f}")
print(f"Expected Return (Swap): ${np.mean(swap_res):.2f}")
print(f"Difference: ${np.mean(swap_res) - np.mean(keep_res):.5f}")


In [ ]:

# %% [markdown]
# ### Reflection
# *Does the simulation agree with the $1.25A$ paradox? Should we change?*
# **Answer:** No. The empirical expected values are identical (approximately $75). The paradox vanishes when a proper probability distribution generates the base amounts. If you observe $A = 150$, the other envelope *must* be $75$, it cannot be $300$ because our base $X$ maxes out at $100$.

# %% [markdown]
# ## Part 2: Parallel Simulation with JAX
#
# In Deep Reinforcement Learning, we cannot rely on Python `for` loops or even standard `numpy` if we want to run millions of environments on a GPU. We use **JAX**.
#
# ### Question 2: Vectorizing with JAX
# Rewrite your simulation using `jax.numpy` and `jax.random`. We will simulate $10,000,000$ games. Notice how we must explicitly pass the pseudo-random number generator (PRNG) `key`.

# %%
def simulate_two_envelopes_jax(key, num_trials):
    """
    A high-performance JAX implementation of the Two Envelopes game.
    """
    # Split the PRNG key for the two random operations
    key_x, key_choice = jax.random.split(key)

    # TODO: Use jax.random.uniform to generate base amounts
    base_amounts = jax.random.uniform(key_x, shape=(num_trials,), minval=1.0, maxval=100.0)

    env_A = base_amounts
    env_B = 2 * base_amounts

    # TODO: Use jax.random.bernoulli to generate choices (True/False)
    choices = jax.random.bernoulli(key_choice, p=0.5, shape=(num_trials,))

    # TODO: Use jnp.where to calculate keep and swap rewards
    keep_rewards = jnp.where(choices, env_B, env_A)
    swap_rewards = jnp.where(choices, env_A, env_B)

    return keep_rewards, swap_rewards

# Compile the function using JIT (Just-In-Time) compilation for maximum speed
jit_simulate = jax.jit(simulate_two_envelopes_jax, static_argnums=(1,))

# Run 10 million environments
key = jax.random.PRNGKey(42)
N_ENVS = 10_000_000

# Execute (the first run compiles it, subsequent runs are lightning fast)
jax_keep, jax_swap = jit_simulate(key, N_ENVS)

print(f"JAX Expected Return (Keep): ${jnp.mean(jax_keep):.4f}")
print(f"JAX Expected Return (Swap): ${jnp.mean(jax_swap):.4f}")

# %% [markdown]
# ## Part 3: Numerically Demonstrating the Central Limit Theorem
#
# The **Central Limit Theorem (CLT)** states that the distribution of the *sample mean* of independent, identically distributed (i.i.d.) random variables approaches a Gaussian (Normal) distribution as the sample size $N$ gets large, regardless of the underlying distribution's shape!
#
# The true distribution of our envelope rewards is weird (it's a mixture of two uniform distributions, $X$ and $2X$). It is **not** a bell curve.
#
# ### Question 3: The Sample Mean Distribution
# Let's group our $10,000,000$ JAX results into $M = 10,000$ batches, each of size $N = 1,000$.
# 1. Calculate the mean of each batch.
# 2. Plot the histogram of these $10,000$ sample means.
# 3. Overlay the theoretical Normal distribution predicted by the CLT.

# %%
# 1. Reshape the 10 million outcomes into a matrix of shape (M, N)
# where M = 10,000 (number of batches) and N = 1,000 (samples per batch)
M = 10000
N = 1000

# TODO: Reshape the jax_keep array
batches = jnp.reshape(jax_keep, (M, N))

# 2. Calculate the sample mean for each batch
# TODO: Compute the mean along the sample axis (axis=1)
sample_means = jnp.mean(batches, axis=1)

# 3. Theoretical parameters for the CLT overlay
# To plot the theoretical Gaussian, we need the true population mean and variance.
population_mean = jnp.mean(jax_keep)
population_std = jnp.std(jax_keep)

# According to the CLT, the standard deviation of the sample mean (Standard Error) is:
# SE = sigma / sqrt(N)
# TODO: Calculate the theoretical standard error
theoretical_se = population_std / jnp.sqrt(N)

# 4. Plotting
plt.figure(figsize=(12, 7))

# Plot the empirical histogram of sample means
plt.hist(np.array(sample_means), bins=60, density=True, alpha=0.6, color='royalblue', edgecolor='black', label='Empirical Distribution of Sample Means')

# Plot the Theoretical Gaussian
x_axis = np.linspace(float(population_mean - 4*theoretical_se), float(population_mean + 4*theoretical_se), 1000)
y_axis = norm.pdf(x_axis, float(population_mean), float(theoretical_se))

plt.plot(x_axis, y_axis, color='crimson', lw=3, label=r'Theoretical CLT Gaussian $\mathcal{N}(\mu, \sigma^2/N)$')

plt.title(f'Central Limit Theorem Demonstration\n(Batch Size N={N})', fontsize=16)
plt.xlabel('Sample Mean of Rewards ($)', fontsize=14)
plt.ylabel('Density', fontsize=14)
plt.legend(fontsize=12)
plt.axvline(float(population_mean), color='black', linestyle='dashed', lw=2, label='True Population Mean')
plt.show()

# %% [markdown]
# ### Final Remark
# Notice how perfectly the empirical histogram matches the theoretical Gaussian curve! Even though the underlying envelope game returns a highly non-Gaussian, flat, mixed-uniform distribution, the **average** of those games forms a perfect bell curve.
#
# In Deep RL, we constantly rely on the CLT. When we use mini-batches to calculate gradients for Policy Gradients (like PPO) or Q-Learning (like DQN), we trust that our batch-averaged gradient is a normally-distributed, reliable estimate of the true direction we need to move in.